# Satellite Data Analysis for Jamaica
## Notebook 4 of 4. Read 45 Years of Kingston Heat

**Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and Technology, University of the West Indies.**

> **STUDENT EDITION.** Cells marked **YOUR TURN** have gaps to fill in. Look
> for `____` and `# TODO`. Many gaps list three options in a comment; one is
> right and the others teach you something by being wrong. Everything else runs
> as given. If you get stuck, read the hint under the cell before asking.
> The day: two hours of missions, then the one-hour Satellite Challenge.

---

# Mission 4: Is Home Getting Hotter? 🌡️

**Your question.** Is Jamaica warming, and what does that actually feel like for
a person who lives here?

**Why it matters.** Heat is no longer just weather. It is a health risk and a
bill: hotter nights the body cannot recover from, longer hours the fans and air
conditioning have to run, harder days for anyone who works outside.

**Your objective.** Find the hottest day Kingston saw in July 2026, and how far
above a normal July it sat. Fastest correct answer takes the whole session.

Two questions carry this notebook. Is Jamaica getting hotter? Yes, and you will
have the number in ten minutes. And how hot did July 2026 actually get? That
second one is your mission.

### What you will be able to do by the end

1. Pull forty-five years of climate records for anywhere on Earth, free
2. Fit a warming trend and state its uncertainty
3. Work out when your data is too coarse for the question you asked

**Time in class:** about 20 minutes. Then the Satellite Challenge hour begins.
**Before you start:** Notebooks 1 to 3.

### Words for this notebook

| Word | What it means here |
|---|---|
| trend | The steady direction underneath the year-to-year wiggle |
| uncertainty | How far an answer could reasonably be wrong |
| reanalysis | A weather model rerun over the past, corrected by real measurements |
| cluster | A group of similar pixels a machine found on its own |
| training data | Labelled examples a model learns from |
| extrapolation | Extending a pattern beyond the data that made it |

---

## Part 1. Setting up

In [ ]:
# Run this once. On Google Colab it takes about a minute.
# If a package is already there, pip will say so and move on.
!pip install -q rasterio requests imageio pandas scikit-learn matplotlib pillow

print("Packages ready.")

In [ ]:
# 🚚 JUST RUN THIS CELL. Nothing to change. It is the toolbox for the whole course.
# ============================================================================
#  JAMAICA EARTH OBSERVATION TOOLKIT
#  Run this cell in every session. It sets up the connection to the satellite
#  archive and defines the handful of functions the whole course uses.
# ============================================================================
import os, math, json, time, warnings
warnings.filterwarnings("ignore")

# GDAL reads the satellite files straight off Amazon's servers over the
# internet. These settings tell it how to behave: no login needed, do not list
# whole directories, retry if the network hiccups.
os.environ.update({
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
})

import requests, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.vrt import WarpedVRT
from PIL import Image, ImageDraw

STAC_URL = "https://earth-search.aws.element84.com/v1/search"

def _stac_post(url, body, timeout=60, tries=4):
    """POST to the archive, retrying politely if the server is having a moment."""
    for attempt in range(tries):
        try:
            r = requests.post(url, json=body, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))    # 2 s, 4 s, 6 s between tries


# House style for every chart in this course.
CYAN, INK, SAND = "#00b8d4", "#12232e", "#e0a458"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 13,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})

# Places in Jamaica used through the course, as [west, south, east, north].
PLACES = {
    "kingston":     [-76.86, 17.93, -76.80, 18.02],
    "black_river":  [-77.90, 17.96, -77.78, 18.08],
    "negril":       [-78.375, 18.25, -78.320, 18.36],
    "montego_bay":  [-77.97, 18.44, -77.88, 18.51],
    "new_hope":     [-78.20, 18.13, -78.08, 18.24],
    "st_elizabeth": [-77.75, 18.00, -77.65, 18.10],
    "portland":     [-76.45, 18.10, -76.32, 18.20],
    "jamaica":      [-78.45, 17.66, -76.15, 18.55],
}

def search_scenes(bbox, start, end, max_cloud=30, limit=50, sort_by="eo:cloud_cover",
                  min_cloud=None, descending=False):
    """Ask the archive which Sentinel-2 pictures exist over a box and a date range.

    Returns a list of STAC 'items'. Each item is a dictionary of metadata plus
    links to the actual image files. Nothing is downloaded yet.

    Set `min_cloud` when you deliberately want a cloudy scene, which is useful
    for testing that your cloud masking actually works.
    """
    cloud_filter = {"lt": max_cloud}
    if min_cloud is not None:
        cloud_filter["gt"] = min_cloud
    query = {
        "collections": ["sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": cloud_filter},
        "limit": limit,
        "sortby": [{"field": f"properties.{sort_by}",
                    "direction": "desc" if descending else "asc"}],
    }
    r = _stac_post(STAC_URL, query, timeout=60)
    return r.json()["features"]

def search_all(bbox, start, end, max_cloud=100, page_size=100, max_pages=20):
    """Every matching scene, not just the first page.

    The archive hands back at most 200 results per request and does not warn you
    that it stopped. This follows the 'next' link until the results run out.
    """
    body = {
        "collections": ["sentinel-2-l2a"], "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": {"lt": max_cloud}},
        "limit": page_size,
        "sortby": [{"field": "properties.datetime", "direction": "asc"}],
    }
    items, url, pages, matched = [], STAC_URL, 0, None
    while url and pages < max_pages:
        r = _stac_post(url, body, timeout=90)
        j = r.json()
        items += j.get("features", [])
        matched = j.get("context", {}).get("matched", matched)
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        pages += 1
        if not nxt:
            break
        url = nxt[0]["href"]
        body = nxt[0].get("body", body)
    if matched and len(items) < matched:
        print(f"Warning: got {len(items)} of {matched}. Raise max_pages.")
    return items

def make_grid(bbox, metres=20):
    """Define a fixed grid of pixels over a box, in plain latitude and longitude.

    Every image we read gets warped onto this same grid. That is what lets us
    subtract a November picture from an October one pixel by pixel, even when
    the two came from different satellite tiles in different map projections.
    """
    lon0, lat0, lon1, lat1 = bbox
    shrink = math.cos(math.radians((lat0 + lat1) / 2))
    width  = int(round((lon1 - lon0) * 111320 * shrink / metres))
    height = int(round((lat1 - lat0) * 110540 / metres))
    transform = transform_from_bounds(lon0, lat0, lon1, lat1, width, height)
    return {"width": width, "height": height, "transform": transform,
            "metres": metres, "bbox": bbox,
            "pixel_hectares": (metres * metres) / 10000.0}

def read_band(item, band, grid, resampling=Resampling.bilinear):
    """Read one colour band of one scene onto our grid. Returns raw integers."""
    with rasterio.open(item["assets"][band]["href"]) as src:
        with WarpedVRT(src, crs="EPSG:4326", transform=grid["transform"],
                       width=grid["width"], height=grid["height"],
                       resampling=resampling) as vrt:
            return vrt.read(1)

def read_reflectance(item, band, grid):
    """Read a band and convert to reflectance (0 to 1). Divide by 10000."""
    return read_band(item, band, grid).astype("float32") / 10000.0

# Scene Classification Layer codes that mean 'this pixel is usable'.
# 4 vegetation, 5 bare soil, 6 water, 7 low-probability cloud, 11 snow/ice.
CLEAR_CODES = [4, 5, 6, 7, 11]

def clear_mask(item, grid):
    """True where the pixel is usable, False where it is cloud, shadow or edge."""
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return np.isin(scl, CLEAR_CODES)

def check_coverage(item, grid):
    """How much of OUR area this scene actually covers, and how much is clear.

    The cloud percentage in the metadata describes the whole 110 km tile. It
    says nothing about your study area. Always check your own box.
    """
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return {"covered": float((scl > 0).mean()),
            "clear": float(np.isin(scl, CLEAR_CODES).mean())}

def best_scene(items, grid, min_covered=0.95, min_clear=0.60, check_n=8):
    """Walk down the candidate list and return the first scene that is genuinely
    good over our box, not just good on paper."""
    for item in items[:check_n]:
        try:
            c = check_coverage(item, grid)
        except Exception:
            continue
        if c["covered"] >= min_covered and c["clear"] >= min_clear:
            item["_coverage"] = c
            return item
    return None

def composite(items, grid, bands, max_scenes=12, min_clear=0.10, verbose=True):
    """Stack several cloud-masked scenes and take the middle value per pixel.

    One picture of Jamaica almost always has cloud somewhere. Stack ten and take
    the median and the clouds disappear, because cloud is bright and rare while
    the ground underneath is consistent.
    """
    stacks = {b: [] for b in bands}
    used = []
    for item in items:
        if len(used) >= max_scenes:
            break
        try:
            clear = clear_mask(item, grid)
            if clear.mean() < min_clear:
                continue
            for b in bands:
                a = read_reflectance(item, b, grid)
                a[~clear] = np.nan
                a[a <= 0] = np.nan
                stacks[b].append(a)
            used.append(item["properties"]["datetime"][:10])
        except Exception:
            continue
    if not used:
        raise RuntimeError("No usable scenes found. Widen the dates or raise max_cloud.")
    if verbose:
        print(f"Composite built from {len(used)} scenes: {', '.join(sorted(used))}")
    out = {b: np.nanmedian(np.stack(v), axis=0) for b, v in stacks.items()}
    out["_dates"] = sorted(used)
    return out

def normalized_difference(a, b):
    """(a - b) / (a + b). The workhorse formula behind every index in this course."""
    return (a - b) / (a + b + 1e-10)

def stretch(rgb, low=2, high=98):
    """Rescale each colour channel so the picture is bright enough to look at."""
    out = np.zeros_like(rgb, dtype="float32")
    for i in range(rgb.shape[2]):
        band = rgb[:, :, i]
        p1, p2 = np.nanpercentile(band, [low, high])
        out[:, :, i] = np.clip((band - p1) / (p2 - p1 + 1e-9), 0, 1)
    return np.nan_to_num(out)

def show(image, title="", cmap=None, vmin=None, vmax=None, bar=False, size=(9, 8)):
    """Draw an array on screen with sensible defaults."""
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if bar:
        fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.show()

def area_hectares(mask, grid):
    """Convert a True/False mask into hectares on the ground."""
    return float(np.nansum(mask)) * grid["pixel_hectares"]

def label_frame(image_uint8, text):
    """Stamp a label bar onto one animation frame, so every frame says what it is."""
    img = Image.fromarray(image_uint8)
    draw = ImageDraw.Draw(img)
    bar = min(14 + 8 * len(text), img.width)
    draw.rectangle([0, 0, bar, 24], fill=(0, 0, 0))
    draw.text((7, 6), text, fill=(255, 255, 255))
    return np.array(img)

def save_gif(frames, path, ms=900):
    """Write labelled frames out as an animated GIF that loops forever."""
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, duration=ms, loop=0)
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(frames)} frames)")

def show_gif(path):
    """Play a GIF inside the notebook."""
    try:
        from IPython.display import Image as _Gif, display
        display(_Gif(filename=path))
    except Exception:
        print("Open the file from the folder panel on the left to watch it.")

print("Toolkit loaded. Study areas available:", ", ".join(PLACES))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Machine learning tools ready.")

---

## Part 2. Forty-five years of Kingston weather, for free

NASA POWER (Prediction Of Worldwide Energy Resources) hands out weather records back to 1981 for any point on Earth. No
account, no key, no cost. Give it a coordinate, get the weather since before
your parents left school.

| Code | What it is |
|---|---|
| `T2M` | Air temperature two metres above the ground, in Celsius |
| `T2M_MAX` | Daily maximum temperature |
| `PRECTOTCORR` | Rainfall, millimetres per day |
| `ALLSKY_SFC_SW_DWN` | Sunlight reaching the ground |

One quirk to remember: in monthly data, month `13` is not a month. It is the
annual figure for that year.

**Definition.** POWER is a *reanalysis*: a weather model run over the entire
past and corrected at every step by satellite and station measurements (NASA's weather-history model, called MERRA-2). That is how it can give a number for
places that never had a weather station.

In [ ]:
def nasa_power(lat, lon, start=1981, end=2025,
               parameters="T2M,T2M_MAX,PRECTOTCORR"):
    """Monthly climate records for one point. Returns a tidy DataFrame."""
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    r = requests.get(url, params={
        "parameters": parameters, "community": "AG",
        "latitude": lat, "longitude": lon,
        "start": start, "end": end, "format": "JSON"}, timeout=120)
    r.raise_for_status()
    block = r.json()["properties"]["parameter"]

    rows = []
    for key in block["T2M"]:
        row = {"year": int(key[:4]), "month": key[4:]}
        for p in block:
            row[p] = block[p][key]
        rows.append(row)
    df = pd.DataFrame(rows)
    return df[(df[list(block)] > -900).all(axis=1)]


def nasa_power_daily(lat, lon, start, end, parameters="T2M,T2M_MAX"):
    # daily records for one point, dates as YYYYMMDD; returns a tidy DataFrame
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    r = requests.get(url, params={
        "parameters": parameters, "community": "AG",
        "latitude": lat, "longitude": lon,
        "start": start, "end": end, "format": "JSON"}, timeout=120)
    r.raise_for_status()
    block = r.json()["properties"]["parameter"]
    first = parameters.split(",")[0]
    rows = [{"date": k, **{p: block[p][k] for p in block}} for k in block[first]]
    df = pd.DataFrame(rows)
    return df[(df[list(block)] > -900).all(axis=1)]

KINGSTON_LAT, KINGSTON_LON = 17.9714, -76.7936
kgn = nasa_power(KINGSTON_LAT, KINGSTON_LON)

annual = kgn[kgn.month == "13"].sort_values("year").reset_index(drop=True)
monthly = kgn[kgn.month != "13"].copy()
monthly["month_num"] = monthly.month.astype(int)

print(f"{len(annual)} years of records, {annual.year.min()} to {annual.year.max()}")
annual.head()

### YOUR TURN 1

Fit a straight line through the annual temperatures and report the warming rate.

`np.polyfit(x, y, 1)` fits a line and returns two numbers: the slope first, then
the intercept.

In [ ]:
x = annual.year.values
y = annual.T2M.values

# TODO: fit a straight line and pull out the slope.
# Options:  np.polyfit(x, y, 1)  /  np.polyfit(y, x, 1)  /  np.mean(y)
slope, intercept = ____

per_decade = slope * 10
total = slope * (x.max() - x.min())

print(f"Warming rate : {slope:+.4f} degrees C per year")
print(f"Per decade   : {per_decade:+.3f} degrees C")
print(f"Since {x.min()}   : {total:+.2f} degrees C")

assert slope > 0, "If this is negative, check the order of your arguments."
print(f"\nFirst decade average : {y[:10].mean():.2f} C")
print(f"Last decade average  : {y[-10:].mean():.2f} C")
print(f"Difference           : {y[-10:].mean() - y[:10].mean():+.2f} C")

*Hint: `np.polyfit(x, y, 1)`. The 1 means a straight line.*

In [ ]:
residuals = y - (slope * x + intercept)
resid_sd = residuals.std(ddof=2)
slope_se = resid_sd / (x.std(ddof=0) * np.sqrt(len(x)))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={"width_ratios": [1.5, 1]})

a1.plot(x, y, "o-", color=CYAN, ms=5, lw=1.2, label="Annual average")
a1.plot(x, slope * x + intercept, "--", color=INK, lw=2.2,
        label=f"{per_decade:+.2f} C per decade")
a1.fill_between(x, slope * x + intercept - resid_sd, slope * x + intercept + resid_sd,
                color=INK, alpha=0.10, label="Year-to-year variation")
a1.set_xlabel("Year"); a1.set_ylabel("Temperature (C)")
a1.set_title("Kingston air temperature, 1981 to 2025")
a1.legend(frameon=False, loc="upper left")

decades = annual.copy()
decades["decade"] = (decades.year // 10) * 10
dec = decades.groupby("decade").T2M.mean()
a2.bar(dec.index.astype(str), dec.values, color=CYAN, width=6)
a2.set_ylim(dec.min() - 0.4, dec.max() + 0.2)
a2.set_ylabel("Average temperature (C)"); a2.set_title("By decade")
for i, v in enumerate(dec.values):
    a2.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout(); plt.show()

print(f"Trend {slope:+.4f} plus or minus {slope_se:.4f} C per year")
print(f"Trend divided by its uncertainty: {slope / slope_se:.1f}")

Compare that last number against the Negril shoreline in Notebook 3, where the
same ratio came out near 1 and we refused to claim a trend.

Here it is far above 2. The warming is larger than the noise it sits in, by a
wide margin, and stating it as a finding is honest. Same test, opposite verdict,
and the test is what makes both verdicts trustworthy.

---

## Part 3. Resolution: one value stands for a 55 km square

Blue Mountain Peak stands 2,256 m up. Mountain air cools about 6.5 degrees per
kilometre of height, so the peak should run 14 degrees cooler than Kingston.
Ask the data and see.

In [ ]:
places = {
    "Kingston":           (17.9714, -76.7936),
    "Blue Mountain Peak": (18.0450, -76.5850),
    "Black River":        (18.0260, -77.8510),
    "Negril":             (18.3000, -78.3500),
    "Montego Bay":        (18.4700, -77.9200),
}

summary = []
for name, (la, lo) in places.items():
    df = nasa_power(la, lo, parameters="T2M")
    ann = df[df.month == "13"].sort_values("year")
    m, b = np.polyfit(ann.year, ann.T2M, 1)
    summary.append({"place": name,
                    "mean_C": ann.T2M.mean(),
                    "per_decade": m * 10,
                    "total_since_1981": m * (ann.year.max() - ann.year.min())})

heat = pd.DataFrame(summary)
heat.round(3)

Kingston and Blue Mountain Peak come back **identical**, to every decimal
place. The peak does not exist in this dataset: POWER's cells are roughly 55 km
by 65 km, and the city, the mountains and the harbour all fall in one cell.
The number is an average over the lot.

In [ ]:
grid_url = "https://power.larc.nasa.gov/api/temporal/climatology/regional"
r = requests.get(grid_url, params={
    "parameters": "T2M", "community": "AG",
    "latitude-min": 16.5, "latitude-max": 19.5,
    "longitude-min": -79.5, "longitude-max": -75.5,
    "format": "JSON", "start": 2001, "end": 2020}, timeout=180)
r.raise_for_status()

cells = []
for feature in r.json()["features"]:
    lon, lat = feature["geometry"]["coordinates"][:2]
    cells.append({"lon": lon, "lat": lat,
                  "T2M": feature["properties"]["parameter"]["T2M"]["ANN"]})
cells = pd.DataFrame(cells)

over_jamaica = cells[(cells.lon.between(-78.6, -76.1)) & (cells.lat.between(17.9, 18.7))]
print(f"Grid cells covering the whole island: {len(over_jamaica)}")
print(f"Cell spacing: {sorted(cells.lat.unique())[1] - sorted(cells.lat.unique())[0]:.3f} "
      f"degrees north-south, "
      f"{sorted(cells.lon.unique())[1] - sorted(cells.lon.unique())[0]:.3f} east-west")

fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(cells.lon, cells.lat, c=cells.T2M, s=1400, marker="s",
                cmap="inferno", vmin=25.5, vmax=28.5)
ax.plot([-78.4, -76.2, -76.2, -78.4, -78.4], [17.7, 17.7, 18.5, 18.5, 17.7],
        color=CYAN, lw=2.5, label="Jamaica, roughly")
ax.scatter([-76.7936], [17.9714], color="white", s=70, zorder=5, label="Kingston")
ax.set_xlim(-79.6, -75.4); ax.set_ylim(16.3, 19.7)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Every temperature NASA POWER has for this part of the Caribbean")
ax.legend(frameon=False, loc="upper right")
fig.colorbar(sc, ax=ax, label="Average temperature (C)")
plt.tight_layout(); plt.show()

The entire island is eight numbers. Enough to ask *is Jamaica warming*.
Useless for *is Half Way Tree hotter than Hope Gardens*. Both are heat
questions; only one fits this data. Part 4 does what our data can do: map the
surfaces that make cities hot, rather than the heat itself.

---

## 📍 Your spot: has it warmed?

Same spot, forty-five years of records, one number. Seconds.

In [ ]:
MY_LAT, MY_LON = ____, ____            # your spot, one last time

df = nasa_power(MY_LAT, MY_LON, parameters="T2M")
ann_s = df[df.month == "13"].sort_values("year")

m_s, _ = np.polyfit(ann_s.year, ann_s.T2M, 1)         # slope of the trend line
total_s = m_s * (ann_s.year.max() - ann_s.year.min())
print(f"Your spot warmed {total_s:+.2f} C since {ann_s.year.min()}")

---

## 🏁 Mission objective: how hot did it get, and how far above normal?

Kingston runs on people who work outside and sleep without air conditioning. For
them, heat is not weather. It is health, and it is money: hotter nights the body
never recovers from, longer hours the fans have to run.

**Find the value.** What was the hottest single day in Kingston in July 2026, and
how far above a normal July did it sit?

Complete the two blanks, run it, and call out the temperature. Fastest correct
answer takes the session.

In [ ]:
# NASA POWER daily records for Kingston, every day of July 2026
july = nasa_power_daily(17.98, -76.80, "20260701", "20260731")

# Blank 1, the column: which one is the daily MAXIMUM temperature? Options: T2M / T2M_MAX
# Blank 2, hottest: do you take the max or the mean across the month? Options: max / mean
hottest = july["____"].____()
print("THE VALUE:", round(hottest, 1), "C")

# how that compares to a normal July (this part is done for you)
normal = nasa_power(17.98, -76.80, parameters="T2M,T2M_MAX")
usual = normal[(normal.month == "07") & (normal.year <= 2025)].T2M_MAX.mean()
print(f"A normal July day tops out near {usual:.1f} C")
print(f"So the hottest day sat {hottest - usual:+.1f} C above normal")

*Stuck? Two of NASA's columns are temperatures. One is the average across the
whole day, the other is the peak the day reached. The word "hottest" points at
the peak. And across a month, the hottest day is the single highest value, not
the typical one.*

---

## What you learned across four notebooks

You started by turning a place into four numbers. You finished by measuring a
Category 5 hurricane and reading 45 years of heat with honest error bars.

The technical skills are worth having. The habits are worth more:

- Check the data covers what you think it covers
- Check you have all of it
- Compare against a period when nothing happened
- Compare the effect against the noise it sits in
- Say what your method cannot see

Every dataset used in this course is free and open. Sentinel-2 has photographed
Jamaica every five days since 2015 and will keep doing so. NASA POWER goes back
to 1981 and updates continuously. No permission is required and no fee is
charged. What was missing was somebody local asking the right questions of it.

---

### Before you close this notebook

Save a copy to your own Drive (`File` then `Save a copy in Drive`). Next up:
the Satellite Challenge. Your team will need everything you just learned.

*Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and
Technology, University of the West Indies.*

*Satellite Data Analysis for Jamaica. Built with free, open data: Sentinel-2 from
the European Space Agency, hosted by Amazon; NASA POWER climate records. No API
keys, no fees, no permission needed.*